In [ ]:
!git clone https://github.com/AutoCS-wyh/Automotive-cyber-threat-intelligence-corpus.git
%cd Automotive-cyber-threat-intelligence-corpus
!ls

Cloning into 'Automotive-cyber-threat-intelligence-corpus'...
remote: Enumerating objects: 3083, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (543/543), done.
remote: Total 3083 (delta 145), reused 61 (delta 8), pack-reused 2530 (from 1)
Receiving objects: 100% (3083/3083), 1.34 MiB | 12.73 MiB/s, done.
Resolving deltas: 100% (467/467), done.
/content/Automotive-cyber-threat-intelligence-corpus
 BIOES			      BIOES.txt   LICENSE   README.md
'BIOES joint annotation.py'   data	  model     read.py


In [ ]:
!pip install seqeval


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1d7f2166ee861267b5efe04efa7852b250b7189aad7e907b26082919e54aac9d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# BiLSTM-att-CRF on ACTI corpus (pure PyTorch, no torchcrf) + 10 iterations
# ------------------------------------------------------------------------
# - Uses tag.dic, train.txt, dev.txt (ACTI format)
# - BiLSTM encoder + global additive attention + custom CRF
# - seqeval for Precision / Recall / F1 / Accuracy
# - Runs 10 full training iterations (different seeds) and prints mean ± std
#
# Colab tip:
#   !pip -q install seqeval

import os
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
)

# ============================================
# 1) Paths to ACTI files  (CHANGE THESE!)
# ============================================
TAG_DIC_PATH = "model/BiLSTM-dynamic-att-LSTM/data1/tag.dic"
TRAIN_PATH   = "model/BiLSTM-dynamic-att-LSTM/data1/train.txt"
DEV_PATH     = "model/BiLSTM-dynamic-att-LSTM/data1/dev.txt"  # used as test set here


# ============================================
# 2) Reproducibility
# ============================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # deterministic (slightly slower)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================
# 3) Load label dictionary
# ============================================
def load_labels(path):
    with open(path, encoding="utf-8") as f:
        labs = [line.strip() for line in f if line.strip()]
    return labs

labels = load_labels(TAG_DIC_PATH)
num_labels = len(labels)
label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for i, lab in enumerate(labels)}

print("Number of labels:", num_labels)


# ============================================
# 4) Load ACTI train/dev (tokens + labels)
#    Each line: "token1 token2 ...\tlabel1 label2 ..."
# ============================================
def load_acti_file(path):
    sentences = []
    tags = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            sent_str, tag_str = parts
            tokens = sent_str.split()
            labs   = tag_str.split()
            if len(tokens) != len(labs):
                raise ValueError(f"Token/label mismatch in {path}")
            if len(tokens) == 0:
                continue
            sentences.append(tokens)
            tags.append(labs)
    return sentences, tags

train_tokens, train_labels = load_acti_file(TRAIN_PATH)
dev_tokens,   dev_labels   = load_acti_file(DEV_PATH)

print("Train sentences:", len(train_tokens))
print("Dev sentences:  ", len(dev_tokens))


# ============================================
# 5) Build word vocabulary from training data
# ============================================
word_counter = Counter()
for sent in train_tokens:
    word_counter.update(sent)

word_vocab = {"<PAD>": 0, "<UNK>": 1}
for w in word_counter.keys():
    if w not in word_vocab:
        word_vocab[w] = len(word_vocab)

vocab_size = len(word_vocab)
print("Word vocab size:", vocab_size)


# ============================================
# 6) Dataset + collate_fn
# ============================================
class ActiDataset(Dataset):
    def __init__(self, sentences, labels, word_vocab, label2id):
        self.sentences = sentences
        self.labels = labels
        self.word_vocab = word_vocab
        self.label2id = label2id

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        tokens = self.sentences[idx]
        labs   = self.labels[idx]

        token_ids = [self.word_vocab.get(t, self.word_vocab["<UNK>"]) for t in tokens]
        label_ids = [self.label2id[l] for l in labs]

        return {
            "input_ids": torch.tensor(token_ids, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
        }

def make_collate_fn(word_vocab, pad_label_id=0):
    pad_id = word_vocab["<PAD>"]

    def collate_fn(batch):
        input_seqs = [item["input_ids"] for item in batch]
        label_seqs = [item["labels"] for item in batch]

        lengths = [len(seq) for seq in input_seqs]
        max_len = max(lengths)

        padded_inputs, padded_labels, masks = [], [], []

        for inp, lab in zip(input_seqs, label_seqs):
            seq_len = len(inp)
            pad_len = max_len - seq_len

            padded_inp = torch.cat([inp, torch.full((pad_len,), pad_id, dtype=torch.long)])
            padded_lab = torch.cat([lab, torch.full((pad_len,), pad_label_id, dtype=torch.long)])

            mask = torch.cat([
                torch.ones(seq_len, dtype=torch.bool),
                torch.zeros(pad_len, dtype=torch.bool)
            ])

            padded_inputs.append(padded_inp)
            padded_labels.append(padded_lab)
            masks.append(mask)

        return {
            "input_ids": torch.stack(padded_inputs),  # [B, T]
            "labels": torch.stack(padded_labels),     # [B, T]
            "mask": torch.stack(masks),               # [B, T] bool
        }

    return collate_fn


train_dataset = ActiDataset(train_tokens, train_labels, word_vocab, label2id)
dev_dataset   = ActiDataset(dev_tokens,   dev_labels,   word_vocab, label2id)


# ============================================
# 7) Custom CRF implementation (pure PyTorch)
# ============================================
class CRF(nn.Module):
    def __init__(self, num_tags: int):
        super().__init__()
        self.num_tags = num_tags

        # transitions[i, j] = score of transitioning from i -> j
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions   = nn.Parameter(torch.empty(num_tags))
        self.transitions       = nn.Parameter(torch.empty(num_tags, num_tags))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions,   -0.1, 0.1)
        nn.init.uniform_(self.transitions,       -0.1, 0.1)

    @staticmethod
    def _log_sum_exp(tensor, dim=-1):
        max_score, _ = tensor.max(dim)
        max_score_broadcast = max_score.unsqueeze(dim)
        return max_score + torch.log(torch.sum(torch.exp(tensor - max_score_broadcast), dim))

    def forward(self, emissions, tags, mask, reduction="mean"):
        """
        log-likelihood of given sequences.
        emissions: [B, T, C]
        tags:      [B, T]
        mask:      [B, T] bool
        """
        llh = self._compute_log_likelihood(emissions, tags, mask)
        if reduction == "mean":
            return llh.mean()
        if reduction == "sum":
            return llh.sum()
        return llh

    def _compute_log_likelihood(self, emissions, tags, mask):
        B, T, C = emissions.shape
        assert C == self.num_tags

        mask_f = mask.float()

        # ---- numerator: score of the provided tag path ----
        score = self.start_transitions[tags[:, 0]]
        score = score + emissions[:, 0, :].gather(1, tags[:, 0].unsqueeze(1)).squeeze(1)

        for t in range(1, T):
            prev_tags = tags[:, t - 1]
            curr_tags = tags[:, t]

            trans_score = self.transitions[prev_tags, curr_tags]
            emit_score  = emissions[:, t, :].gather(1, curr_tags.unsqueeze(1)).squeeze(1)

            score = score + (trans_score + emit_score) * mask_f[:, t]

        # end transitions (use last real tag)
        seq_lens = mask.long().sum(dim=1)  # [B]
        last_idx = (seq_lens - 1).clamp(min=0)
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last_tags]

        # ---- denominator: log-partition ----
        partition = self._compute_log_partition(emissions, mask)
        return score - partition

    def _compute_log_partition(self, emissions, mask):
        B, T, C = emissions.shape
        # alpha: [B, C]
        alpha = self.start_transitions + emissions[:, 0, :]

        for t in range(1, T):
            emit_t = emissions[:, t, :]      # [B, C]
            mask_t = mask[:, t].unsqueeze(1) # [B, 1] bool

            # scores: [B, C_from, C_to]
            scores = alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + emit_t.unsqueeze(1)
            new_alpha = self._log_sum_exp(scores, dim=1)  # [B, C_to]

            alpha = torch.where(mask_t, new_alpha, alpha)

        alpha = alpha + self.end_transitions
        return self._log_sum_exp(alpha, dim=1)  # [B]

    def decode(self, emissions, mask):
        """
        Viterbi decode.
        returns list[list[int]] of best tag ids per sequence
        """
        B, T, C = emissions.shape
        score = self.start_transitions + emissions[:, 0, :]  # [B, C]
        history = []  # list of backpointers [B, C]

        for t in range(1, T):
            emit_t = emissions[:, t, :]      # [B, C]
            mask_t = mask[:, t].unsqueeze(1) # [B, 1] bool

            next_score = score.unsqueeze(2) + self.transitions.unsqueeze(0)  # [B, C_from, C_to]
            max_score, max_tag = next_score.max(dim=1)                       # [B, C_to], [B, C_to]
            max_score = max_score + emit_t                                   # add emission

            score = torch.where(mask_t, max_score, score)
            history.append(max_tag)

        score = score + self.end_transitions
        best_last_score, best_last_tag = score.max(dim=1)  # [B]

        # backtrack
        seq_lens = mask.long().sum(dim=1)  # [B]
        best_paths = []

        for i in range(B):
            length = int(seq_lens[i].item())
            last_tag = int(best_last_tag[i].item())
            path = [last_tag]

            # history has length T-1, we only backtrack length-1 steps
            for hist_t in range(length - 2, -1, -1):
                last_tag = int(history[hist_t][i, last_tag].item())
                path.append(last_tag)

            path.reverse()
            best_paths.append(path)

        return best_paths


# ============================================
# 8) BiLSTM-att-CRF model
# ============================================
class BiLSTMAttCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        emb_dim=300,
        hidden_dim=300,
        attn_hidden_dim=128,
        dropout=0.1,
    ):
        super().__init__()
        self.num_labels = num_labels

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.bilstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        # global additive attention
        self.attn_layer = nn.Sequential(
            nn.Linear(hidden_dim, attn_hidden_dim),
            nn.Tanh(),
            nn.Linear(attn_hidden_dim, 1),
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

        self.crf = CRF(num_labels)

    def _compute_emissions(self, input_ids, mask):
        emb = self.embedding(input_ids)     # [B, T, E]
        enc_out, _ = self.bilstm(emb)       # [B, T, H]

        attn_scores = self.attn_layer(enc_out).squeeze(-1)      # [B, T]
        attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=-1)       # [B, T]

        context = torch.bmm(attn_weights.unsqueeze(1), enc_out).squeeze(1)  # [B, H]
        context_expanded = context.unsqueeze(1).expand(-1, enc_out.size(1), -1)  # [B, T, H]

        combined = torch.cat([enc_out, context_expanded], dim=-1)  # [B, T, 2H]
        emissions = self.fc(self.dropout(combined))                # [B, T, num_labels]
        return emissions

    def forward(self, input_ids, labels=None, mask=None):
        emissions = self._compute_emissions(input_ids, mask)
        if labels is not None:
            llh = self.crf(emissions, labels, mask, reduction="mean")
            return -llh  # negative log-likelihood
        return self.crf.decode(emissions, mask)


# ============================================
# 9) Evaluation with seqeval
# ============================================
def evaluate(model, data_loader, device, print_report=False):
    model.eval()
    seqeval_true, seqeval_pred = [], []
    total_loss, steps = 0.0, 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)
            mask      = batch["mask"].to(device)

            loss = model(input_ids, labels=labels, mask=mask)
            total_loss += loss.item()
            steps += 1

            pred_paths = model(input_ids, labels=None, mask=mask)  # list[list[int]]

            B, T = labels.shape
            for i in range(B):
                seq_len = int(mask[i].sum().item())
                if seq_len <= 0:
                    continue

                true_seq = [id2label[int(labels[i, j].item())] for j in range(seq_len)]
                pred_seq = [id2label[int(pred_paths[i][j])] for j in range(seq_len)]
                seqeval_true.append(true_seq)
                seqeval_pred.append(pred_seq)

    avg_loss = total_loss / steps if steps > 0 else 0.0

    ent_precision = precision_score(seqeval_true, seqeval_pred)
    ent_recall    = recall_score(seqeval_true, seqeval_pred)
    ent_f1        = f1_score(seqeval_true, seqeval_pred)
    ent_accuracy  = accuracy_score(seqeval_true, seqeval_pred)

    results = {
        "eval_loss": float(avg_loss),
        "eval_precision": float(ent_precision),
        "eval_recall": float(ent_recall),
        "eval_f1": float(ent_f1),
        "eval_accuracy": float(ent_accuracy),
    }

    if print_report:
        print("\nClassification report:\n")
        print(classification_report(seqeval_true, seqeval_pred))

    return results


# ============================================
# 10) One iteration (train + best dev)
# ============================================
def run_one_iteration(
    seed: int,
    num_epochs: int = 100,
    batch_size: int = 16,
    lr: float = 1e-3,
    grad_clip: float = 5.0,
    use_best_dev: bool = True,   # True => return best dev F1 over epochs
):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # DataLoaders (seed affects shuffle order via generator)
    g = torch.Generator()
    g.manual_seed(seed)

    collate_fn = make_collate_fn(word_vocab, pad_label_id=0)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        generator=g,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = BiLSTMAttCRF(
        vocab_size=vocab_size,
        num_labels=num_labels,
        emb_dim=300,
        hidden_dim=300,
        attn_hidden_dim=128,
        dropout=0.1,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_res = None
    last_res = None

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_train_loss = 0.0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)
            mask      = batch["mask"].to(device)

            optimizer.zero_grad()
            loss = model(input_ids, labels=labels, mask=mask)
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / max(1, len(train_loader))

        dev_res = evaluate(model, dev_loader, device, print_report=False)
        last_res = dev_res

        # track best F1
        if best_res is None or dev_res["eval_f1"] > best_res["eval_f1"]:
            best_res = dev_res

        print(
            f"[seed={seed}] epoch {epoch:02d} | train_loss={avg_train_loss:.4f} | "
            f"dev_f1={dev_res['eval_f1']:.4f} | dev_p={dev_res['eval_precision']:.4f} | dev_r={dev_res['eval_recall']:.4f}"
        )

    out = best_res if use_best_dev else last_res
    out = dict(out)
    out["seed"] = seed
    return out


# ============================================
# 11) Run 10 iterations + summarize
# ============================================
def run_n_iterations(
    n_runs: int = 10,
    seeds=None,
    num_epochs: int = 100,
    batch_size: int = 16,
    lr: float = 1e-3,
    use_best_dev: bool = True,
):
    if seeds is None:
        seeds = list(range(1, n_runs + 1))

    all_results = []
    for s in seeds:
        print("\n" + "=" * 90)
        print(f"RUN {len(all_results)+1}/{len(seeds)} (seed={s})")
        print("=" * 90)
        res = run_one_iteration(
            seed=s,
            num_epochs=num_epochs,
            batch_size=batch_size,
            lr=lr,
            use_best_dev=use_best_dev,
        )
        all_results.append(res)

    keys = ["eval_loss", "eval_precision", "eval_recall", "eval_f1", "eval_accuracy"]

    print("\n" + "#" * 90)
    print("Per-run results")
    print("#" * 90)
    print("seed\t" + "\t".join(keys))
    for r in all_results:
        print(
            f"{r['seed']}\t"
            f"{r['eval_loss']:.6f}\t"
            f"{r['eval_precision']:.6f}\t"
            f"{r['eval_recall']:.6f}\t"
            f"{r['eval_f1']:.6f}\t"
            f"{r['eval_accuracy']:.6f}"
        )

    print("\n" + "#" * 90)
    print("Average over runs (mean ± std)")
    print("#" * 90)
    for k in keys:
        vals = np.array([r[k] for r in all_results], dtype=float)
        mean = vals.mean()
        std = vals.std(ddof=1) if len(vals) > 1 else 0.0
        print(f"{k}: {mean:.6f} ± {std:.6f}")

    return all_results


# ============================================
# 12) Main
# ============================================
if __name__ == "__main__":
    # use_best_dev=True -> each run reports best dev F1 achieved during training
    # set to False -> each run reports FINAL epoch dev metrics
    run_n_iterations(
        n_runs=10,
        num_epochs=20,
        batch_size=16,
        lr=1e-3,
        use_best_dev=True,
    )


Number of labels: 254
Train sentences: 2384
Dev sentences:   596
Word vocab size: 6003

RUN 1/10 (seed=1)
[seed=1] epoch 01 | train_loss=36.3530 | dev_f1=0.0904 | dev_p=0.2652 | dev_r=0.0545
[seed=1] epoch 02 | train_loss=21.6184 | dev_f1=0.2416 | dev_p=0.2833 | dev_r=0.2105
[seed=1] epoch 03 | train_loss=14.7294 | dev_f1=0.3105 | dev_p=0.3761 | dev_r=0.2644
[seed=1] epoch 04 | train_loss=9.9486 | dev_f1=0.3498 | dev_p=0.3801 | dev_r=0.3239
[seed=1] epoch 05 | train_loss=6.4363 | dev_f1=0.3618 | dev_p=0.3966 | dev_r=0.3327
[seed=1] epoch 06 | train_loss=3.9176 | dev_f1=0.3616 | dev_p=0.4014 | dev_r=0.3289
[seed=1] epoch 07 | train_loss=2.4068 | dev_f1=0.3520 | dev_p=0.3820 | dev_r=0.3264
[seed=1] epoch 08 | train_loss=1.6273 | dev_f1=0.3635 | dev_p=0.3917 | dev_r=0.3390
[seed=1] epoch 09 | train_loss=1.1757 | dev_f1=0.3577 | dev_p=0.3867 | dev_r=0.3327
[seed=1] epoch 10 | train_loss=0.8974 | dev_f1=0.3696 | dev_p=0.3945 | dev_r=0.3477
[seed=1] epoch 11 | train_loss=0.6691 | dev_f1=0.35